# EDA Rebuild AGENT

This notebook rebuilds the feature-engineering pipeline in a **readable, step-by-step** way.

Design principles of this notebook:
- one visible function call per logical block;
- raw data is read only from `data/raw`;
- derived outputs are cached in `data/AGENTS_NEW`;
- if a cached table already exists, the function loads it and still shows a user-facing summary and charts;
- merge checks and bridge checks stay explicit instead of being hidden in a monolithic script;
- the final table remains target-free and training-ready at the `users_course_id` grain.

## 1. Notebook Setup

The first cell resolves the project root, imports the modular pipeline functions, and exposes the helper functions used later for explicit validation and merge diagnostics.

In [ ]:
# Resolve the project root so notebook imports work reliably.
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Import notebook-facing entity functions.
from scripts.agents_new_pipeline import (
    run_audit_block,
    build_base_entity_block,
    build_course_block,
    build_user_block,
    build_user_lessons_block,
    build_user_trainings_block,
    build_user_answers_block,
    build_course_actions_block,
    build_media_sessions_block,
    build_access_history_block,
    build_final_feature_table_block,
)

# Import explicit validation and merge helpers.
from scripts.agents_new_pipeline.validation_utils import (
    validate_block_coverage,
    validate_unique_key,
)
from scripts.agents_new_pipeline.merge_utils import merge_feature_block
from scripts.agents_new_pipeline.config import OUT_DIR

OUT_DIR

## 2. Raw Audit and Entity Map

This block audits raw tables before any modeling table is built. It reports table sizes, missingness samples, candidate-key checks, and the entity map used in the rest of the pipeline.

In [ ]:
# Audit the raw layer and cache the diagnostics in data/AGENTS_NEW.
audit_result = run_audit_block()
audit_result.data.head()

## 3. Core Enrollment Entity

The central business entity is **one user on one course**, represented by `users_course_id`.

This block:
- removes agent accounts;
- keeps one row per enrollment;
- preserves course timing anchors;
- avoids target construction;
- exposes uniqueness diagnostics directly in the notebook output.

In [ ]:
# Build the enrollment base that all later blocks will merge into.
base_result = build_base_entity_block()
base_result.data[['users_course_id', 'user_id', 'course_id']].head()

## 4. User-Level Enrichment

User-level features are built separately from the enrollment base so that the grain stays clean.

This block contributes profile-like but still interpretable context features such as sign-in activity, grade, subscription status, geography ids, and badge-event signals.

In [ ]:
# Build user-level enrichment features before any enrollment-level merge.
user_result = build_user_block()
user_result.data.head()

## 5. Course Structure Features

Course structure is built at the `course_id` grain first.

The function aggregates lesson metadata, lesson-task links, trainings, and webinar/group structure.
It also displays reconciliation evidence against course-level signals observed inside the enrollment export.

In [ ]:
# Build course-level structural context before joining it to enrollments.
course_result = build_course_block(base_result=base_result)
course_result.data.head()

## 6. User-Lesson Aggregation

This block aggregates user progress on lessons directly to the enrollment level.

It focuses on lesson visits, solved lessons, lesson points, unique lessons touched, and partial progression signals when lesson ordering is available.

In [ ]:
# Aggregate lesson-level progress to users_course_id.
user_lessons_result = build_user_lessons_block(base_result=base_result)
user_lessons_result.data.head()

## 7. User-Training Aggregation

Training behavior must be resolved through the explicit bridge:
`user_trainings -> trainings -> lessons -> course_id -> users_courses`.

The function returns both the aggregated enrollment-level feature table and the bridge diagnostics so the linkage quality stays visible.

In [ ]:
# Resolve training behavior through course structure before aggregation.
user_trainings_result = build_user_trainings_block(base_result=base_result)
user_trainings_result.data.head()

## 8. User-Answer Aggregation

Answers are not linked by `task_id` directly because that path is ambiguous across courses.

Instead, the function uses `resource_type/resource_id` bridges and then aggregates answer intensity, points, attempts, timing, and answer-type shares to the enrollment grain.

In [ ]:
# Build the answer-behavior block via explicit resource bridges.
user_answers_result = build_user_answers_block(base_result=base_result)
user_answers_result.data.head()

## 9. Course Actions

The action log is the main time-aware behavioral source.

This block captures action volume, action mix, early activity windows, peak timing, active-day counts, and inactivity-gap signals.

In [ ]:
# Aggregate the user-course action log and expose temporal engagement signals.
course_actions_result = build_course_actions_block(base_result=base_result)
course_actions_result.data.head()

## 10. Media Sessions and Access History

Media sessions and access windows are kept in separate blocks because they come from different operational processes and have different native grains.

The media block resolves resources back to courses, while the access block summarizes access intervals, reopen signals, and access gaps.

In [ ]:
# Build media-session features.
media_result = build_media_sessions_block(base_result=base_result)

# Build access-history features.
access_result = build_access_history_block(base_result=base_result)

media_result.data.head(), access_result.data.head()

## 11. Explicit Validation Helpers

The modular pipeline also exposes reusable checking functions.

Below, the notebook explicitly calls a coverage checker and a merge helper so the user can inspect the logic instead of trusting a hidden pipeline.

In [ ]:
# Example 1: coverage validation for the training block against the enrollment base.
training_coverage_check = validate_block_coverage(
    base_df=base_result.data,
    block_df=user_trainings_result.data,
    key_column='users_course_id',
    block_name='user_trainings_manual_check',
)
training_coverage_check

In [ ]:
# Example 2: explicit merge helper on the course block.
course_merge_preview, course_merge_validation = merge_feature_block(
    base_df=base_result.data[['users_course_id', 'course_id']].copy(),
    block_df=course_result.data.copy(),
    block_name='course_features_preview_merge',
    join_key='course_id',
)

course_merge_validation

## 12. Final Assembly

The final block performs the full user-course assembly and returns a training-ready feature table.

Important constraints preserved here:
- one row per `users_course_id`;
- no target construction;
- no model training;
- no raw-file modifications;
- merge diagnostics and feature-quality summaries remain visible.

In [ ]:
# Merge all cached blocks into the final training-ready feature table.
final_result = build_final_feature_table_block(
    base_result=base_result,
    course_result=course_result,
    user_result=user_result,
    user_lessons_result=user_lessons_result,
    user_trainings_result=user_trainings_result,
    user_answers_result=user_answers_result,
    course_actions_result=course_actions_result,
    media_result=media_result,
    access_result=access_result,
)

final_result.data[['users_course_id', 'user_id', 'course_id']].head()

## 13. Final Sanity Checks

The last code cell runs a couple of simple assertions to confirm that the final table still respects the intended grain.

In [ ]:
# Confirm one row per users_course_id.
assert final_result.data['users_course_id'].nunique() == len(final_result.data)
assert final_result.data['users_course_id'].isna().sum() == 0

{
    'rows': len(final_result.data),
    'columns': final_result.data.shape[1],
    'unique_users_course_id': final_result.data['users_course_id'].nunique(),
    'cached_output_dir': str(OUT_DIR),
}